In [1]:
# --- Celda 1.1: Importaciones y Carga de Librerías Offline ---
import os
import numpy as np
import pandas as pd
import glob
import ROOT
import time

# Esta es la parte MÁS IMPORTANTE:
# Asegúrate de que estás corriendo este Jupyter Lab desde una terminal
# donde ANTES hiciste: source /ruta/a/auger/offline/this-auger-offline.sh

AugerOfflineRoot = os.environ.get("AUGEROFFLINEROOT")
if AugerOfflineRoot is None:
    raise EnvironmentError(
        "AUGEROFFLINEROOT no definido. "
        "Reinicia Jupyter Lab desde una terminal donde hayas "
        "hecho: "
        " 'aug_set_version offline 4.0.1-icrc23-prod1-root6' "   
        " 'source /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6/bin/this-auger-offline.sh'."
    )

print(f"AUGEROFFLINEROOT encontrado en: {AugerOfflineRoot}")

# Cargar las librerías necesarias
print("Cargando librerías de Auger Offline...")
libs_to_load = ["libRecEventKG.so"]
for lib in libs_to_load:
    lib_path = os.path.join(AugerOfflineRoot, "lib", lib)
    if not os.path.exists(lib_path):
        raise FileNotFoundError(f"No se encontró la librería: {lib_path}")
    
    # Usamos gSystem.Load que es más robusto en PyROOT
    status = ROOT.gSystem.Load(lib_path)
    if status < 0:
        raise ImportError(f"Error cargando la librería: {lib_path}")

print("Librerías cargadas correctamente. ¡Listo para trabajar! 🚀")

Welcome to JupyROOT 6.30/04
AUGEROFFLINEROOT encontrado en: /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6
Cargando librerías de Auger Offline...
Librerías cargadas correctamente. ¡Listo para trabajar! 🚀


In [2]:
# --- Celda 2.1: Funciones Auxiliares y Principales (Actualizada) ---

def getCounterList(sevent, mevent):
    """
    Obtener la lista de counters UMD asociados a las estaciones SD de un evento.
    """
    cList = []
    stationVector = sevent.GetStationVector()
    for station in stationVector:
        stationId = station.GetId()
        counterId = int("10" + str(stationId))
        if mevent.HasCounter(counterId):
            cList.append(mevent.GetCounter(counterId))
    return cList

def getModuleList(counter, sim=True):
    """
    Obtener la lista de módulos (scintillators) de un counter UMD.
    """
    possibleModules = range(0, 6) if sim else range(100, 116)
    modules = []
    for modId in possibleModules:
        if counter.HasModule(modId):
            modules.append(counter.GetModule(modId))
    return modules

def readADST_surface(fname):
    """
    Leer un archivo ADST y extraer info en un DataFrame "plano".
    Cada fila corresponde a UN counter UMD.
    """
    
    print(f"Iniciando lectura de: {os.path.basename(fname)}")
    
    if not os.path.exists(fname):
        print(f"Advertencia: Archivo no encontrado {fname}")
        return pd.DataFrame() # Retorna DF vacío

    files = ROOT.std.vector('string')()
    files.push_back(fname)

    # Inicialización ADST
    file1 = ROOT.RecEventFile(files)
    event = ROOT.RecEvent()
    geo = ROOT.DetectorGeometry()
    
    # Leemos la geometría. Ignoramos si falla.
    file1.ReadDetectorGeometry(geo)
    file1.SetBuffers(event)

    data = [] # Esta lista contendrá una fila por COUNTER
    event_count = 0
    start_time = time.time()

    while file1.ReadNextEvent() == ROOT.RecEventFile.eSuccess:
        event_count += 1
        if event_count % 500 == 0:
            print(f"... procesados {event_count} eventos.")

        # ----------------- MC -----------------
        MCShower = event.GetGenShower()
        logE_MC = np.log10(MCShower.GetEnergy())
        theta_MC = MCShower.GetZenith() * 180.0 / np.pi
        phi_MC = MCShower.GetAzimuth() * 180.0 / np.pi
        primary = MCShower.GetShortPrimaryName()
        
        # ----------------- REC -----------------
        sEvent = event.GetSDEvent()
        sShower = sEvent.GetSdRecShower()
        logE_REC = np.log10(sShower.GetEnergy())
        theta_REC = sShower.GetZenith() * 180.0 / np.pi
        phi_REC = sShower.GetAzimuth() * 180.0 / np.pi
        
        # ----------------- MD -----------------
        mEvent = event.GetMDEvent()
        counterList = getCounterList(sEvent, mEvent)

        if not counterList:
            continue # Saltamos eventos sin señal UMD

        for counter in counterList:
            if counter.IsRejected() or counter.IsSaturated():
                continue

            # Número total de muones por counter            
            nMuones = counter.GetNumberOfMuons()

            # Coordenadas SD asociadas
            sdId = counter.GetSdPartnerId()
            sdStation = sEvent.GetStationById(sdId) if sEvent.HasStation(sdId) else None
            if sdStation:
                r = sdStation.GetSPDistance()
                phi_rel = sdStation.GetAzimuthSP() - sShower.GetAzimuth()
                x_plane = r * np.cos(phi_rel)
                y_plane = r * np.sin(phi_rel)
                r_core = r
                sdSignal = sdStation.GetTotalSignal()
                # Calculamos el phi en el plano de la lluvia (0 a 2*pi)
                phi_plane = np.arctan2(y_plane, x_plane) % (2 * np.pi) # <-- AÑADIDO
            else:
                x_plane, y_plane, r_core, sdSignal, phi_plane = None, None, None, None, None # <-- AÑADIDO

            data.append({
                # Info MC
                "logE_MC": logE_MC, "theta_MC": theta_MC, "phi_MC": phi_MC, "primary": primary,
                
                # Info REC
                "logE_REC": logE_REC, "theta_REC": theta_REC, "phi_REC": phi_REC,
                
                # Info específica del Counter
                "counterId": counter.GetId(),
                "nMuones": nMuones,
                "x_plane": x_plane,
                "y_plane": y_plane,
                "phi_plane": phi_plane, # <-- AÑADIDO
                "r_core": r_core,
                "sdId": sdId,
                "sdSignal": sdSignal
            })

    end_time = time.time()
    elapsed = end_time - start_time
    print(f"Lectura completa. Total de eventos leídos: {event_count}")
    print(f"Tiempo total de lectura: {elapsed:.2f} segundos.")
    print(f"Total de 'counters' (filas) extraídos: {len(data)}")

    df = pd.DataFrame(data)
    return df

In [3]:
# --- Celda 3.1: Configuración del Test ---

# CAMBIA ESTAS RUTAS A TU GUSTO
base_path = "/home/lsilva/Github/ADST_Marina_prot_175/"
output_dir = os.path.join(base_path, "parquet_output_test") # Directorio de salida para el test

# Asegúrate de que la carpeta de salida exista
os.makedirs(output_dir, exist_ok=True)

# Elige UN archivo para la prueba (ejemplo: el primero de la lista)
# (Actualiza esto con un archivo que sepas que existe)
test_file_name = "ADST_e17m316prot-epos_th00_DAT750960.root"
test_file_path = os.path.join(base_path, test_file_name)

print(f"Archivo de prueba seleccionado:\n{test_file_path}")
if not os.path.exists(test_file_path):
    print("¡¡¡ADVERTENCIA!!! El archivo de prueba no existe. Verifica la ruta.")

Archivo de prueba seleccionado:
/home/lsilva/Github/ADST_Marina_prot_175/ADST_e17m316prot-epos_th00_DAT750960.root


In [4]:
# --- Celda 3.2: Ejecución del Test ---

print("Iniciando prueba de lectura...")
start_test_time = time.time()

# Llamamos a la función de lectura
df_test = readADST_surface(test_file_path)

end_test_time = time.time()
print(f"\nFunción 'readADST_surface' completada en {end_test_time - start_test_time:.2f}s")

if not df_test.empty:
    print(f"\n¡Éxito! Se generó un DataFrame con {len(df_test)} filas (counters) y {len(df_test.columns)} columnas.")
else:
    print("\nEl DataFrame resultante está vacío. Revisa si el archivo tenía datos UMD.")

Iniciando prueba de lectura...
Iniciando lectura de: ADST_e17m316prot-epos_th00_DAT750960.root
Lectura completa. Total de eventos leídos: 1
Tiempo total de lectura: 0.33 segundos.
Total de 'counters' (filas) extraídos: 8

Función 'readADST_surface' completada en 5.53s

¡Éxito! Se generó un DataFrame con 8 filas (counters) y 15 columnas.


  input file /home/lsilva/Github/ADST_Marina_prot_175/ADST_e17m316prot-epos_th00_DAT750960.root
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /home/lsilva/Github/ADST_Marina_prot_175/ADST_e17m316prot-epos_th00_DAT750960.root
   has the same version (=27) as the active class but a different checksum.
   You should update the version to ClassDef(Detector,28).
   Do not try to write objects with the current class definition,
   the files will not be readable.

Warning in <TStreamerInfo::CompareContent>: The following data member of
the on-file layout version 27 of class 'Detector' differs from 
the in-memory layout version 27:
   map<int,EyePointingIdMap> fTelPointingIds; //
vs
   map<int,map<int,TString> > fTelPointingIds; //
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class RdRecStationParameterStorageMap read from file /home/lsilva/Github/ADST_Marina_prot_175/ADST_e17m316prot-epos_th00_DAT750960.root
   has the same ve

In [5]:
# --- Celda 4.1: Inspeccionar el DataFrame ---

if not df_test.empty:
    # Muestra las primeras 5 filas
    print("--- Primeras 5 filas (head) ---")
    print(df_test.head())
    
    print("\n--- Información del DataFrame (info) ---")
    # Muestra un resumen (columnas, tipos de datos, uso de memoria)
    df_test.info()
    
    print("\n--- Resumen estadístico (describe) ---")
    # Muestra estadísticas básicas de las columnas numéricas
    print(df_test.describe())
else:
    print("El DataFrame está vacío, no hay nada que inspeccionar.")

--- Primeras 5 filas (head) ---
     logE_MC      theta_MC      phi_MC primary   logE_REC  theta_REC  \
0  17.500003  2.599192e-14  281.780657       p  17.280496   0.668314   
1  17.500003  2.599192e-14  281.780657       p  17.280496   0.668314   
2  17.500003  2.599192e-14  281.780657       p  17.280496   0.668314   
3  17.500003  2.599192e-14  281.780657       p  17.280496   0.668314   
4  17.500003  2.599192e-14  281.780657       p  17.280496   0.668314   

    phi_REC  counterId     nMuones     x_plane     y_plane  phi_plane  \
0  26.37682     104004  668.602217   14.125997   37.305177   1.208820   
1  26.37682     104011    1.971086  465.104468 -562.464126   5.403322   
2  26.37682     104013    2.972841 -730.778417  -53.413756   3.214554   
3  26.37682     104001    0.000000  308.053324  727.792726   1.170391   
4  26.37682     104005    1.971086 -436.851149  637.073487   2.171871   

       r_core  sdId     sdSignal  
0   39.890099  4004  1989.947378  
1  729.854821  4011     5.

In [6]:
# --- Celda 4.2: Guardar en Parquet ---

if not df_test.empty:
    # Definir el nombre del archivo de salida
    output_parquet_name = test_file_name.replace(".root", ".parquet")
    output_parquet_path = os.path.join(output_dir, output_parquet_name)
    
    print(f"Guardando DataFrame en: {output_parquet_path}")
    
    # Guardar en formato Parquet
    df_test.to_parquet(
        output_parquet_path,
        compression="snappy", # Rápido y buena compresión
        index=False
    )
    
    print("¡Guardado exitosamente!")

    # --- Verificación ---
    print("\nVerificando el archivo guardado...")
    df_leido = pd.read_parquet(output_parquet_path)
    print(f"Archivo Parquet leído de vuelta, contiene {len(df_leido)} filas.")
    print("¡Test completado! 🎉")
    
else:
    print("El DataFrame está vacío, no se guardó ningún archivo.")

Guardando DataFrame en: /home/lsilva/Github/ADST_Marina_prot_175/parquet_output_test/ADST_e17m316prot-epos_th00_DAT750960.parquet
¡Guardado exitosamente!

Verificando el archivo guardado...
Archivo Parquet leído de vuelta, contiene 8 filas.
¡Test completado! 🎉
